本文件简陋地手动模拟了 fit 操作。

In [48]:
!python3 --version

Python 3.11.11


In [49]:
!nvidia-smi

Mon Jun  9 09:09:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             27W /   70W |     117MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2.18.0
sys.version_info(major=3, minor=11, micro=11, releaselevel='final', serial=0)
matplotlib 3.7.2
numpy 1.26.4
pandas 2.2.3
sklearn 1.2.2
tensorflow 2.18.0
keras._tf_keras.keras 3.8.0


In [51]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

if gpus:
    print(f"GPUs available: {gpus}")
    try:
        # 打印每个GPU的详细信息
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # 推荐设置，按需分配显存
            print(f"Details for {gpu.name}:")
            # 可以尝试执行一个小操作来确认
        print("GPU is available and TensorFlow can see it!")

        # 尝试一个简单的GPU运算
        print("\nAttempting a simple computation on GPU...")
        with tf.device('/GPU:0'): # 明确指定在第一个GPU上运行
            a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            c = tf.matmul(a, b)
        print("Matrix multiplication result from GPU:")
        print(c.numpy())
        print("If no errors occurred, GPU is working!")

    except RuntimeError as e:
        print(f"RuntimeError during GPU setup or test: {e}")
else:
    print("GPU not available to TensorFlow. TensorFlow will run on CPU.")

TensorFlow Version: 2.18.0
Num GPUs Available: 2
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Details for /physical_device:GPU:0:
Details for /physical_device:GPU:1:
GPU is available and TensorFlow can see it!

Attempting a simple computation on GPU...
Matrix multiplication result from GPU:
[[ 7. 10.]
 [15. 22.]]
If no errors occurred, GPU is working!


In [52]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing.DESCR)
print(housing.data.shape)
print(housing.target.shape)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

    :Number of Instances: 20640

    :Number of Attributes: 8 numeric, predictive attributes and the target

    :Attribute Information:
        - MedInc        median income in block group
        - HouseAge      median house age in block group
        - AveRooms      average number of rooms per household
        - AveBedrms     average number of bedrooms per household
        - Population    block group population
        - AveOccup      average number of household members
        - Latitude      block group latitude
        - Longitude     block group longitude

    :Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived

In [53]:
from sklearn.model_selection import train_test_split

x_train_all, x_test, y_train_all, y_test = train_test_split(
    housing.data, housing.target, random_state = 7)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_all, y_train_all, random_state = 11)
print(x_train.shape, y_train.shape)
print(x_valid.shape, y_valid.shape)
print(x_test.shape, y_test.shape)


(11610, 8) (11610,)
(3870, 8) (3870,)
(5160, 8) (5160,)


In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_valid_scaled = scaler.transform(x_valid)
x_test_scaled = scaler.transform(x_test)

In [55]:
# 看下 metric(算子) 使用
metric = keras.metrics.MeanSquaredError()  # 均方误差
print(metric([5.], [2.]))
print(metric([0.], [1.]))
#具有累加功能，第1个是9，第二个是1，平均是5，(9+1)/2
print(metric.result())

tf.Tensor(9.0, shape=(), dtype=float32)
tf.Tensor(5.0, shape=(), dtype=float32)
tf.Tensor(5.0, shape=(), dtype=float32)


In [56]:
# 不想累加就reset
metric.reset_state()  # 每次epoch需要reset
metric([1.], [3.])
print(metric.result())

tf.Tensor(4.0, shape=(), dtype=float32)


In [57]:
#训练集的样本数
print(len(x_train_scaled))
print(x_train.shape[1:])  #特征数

11610
(8,)


In [58]:
11610/32  #每次训练给予的样本数，我们是自己训练，不使用 fit ,所以提前算好

362.8125

In [59]:
t= np.arange(6).reshape(1, 2, 1, 3)
print(t)
tf.squeeze(t)  # [2, 3]  #降维，将维数为1去除

[[[[0 1 2]]

  [[3 4 5]]]]


<tf.Tensor: shape=(2, 3), dtype=int64, numpy=
array([[0, 1, 2],
       [3, 4, 5]])>

In [60]:
#随机挑选5个样本，看下特征，标签
idx = np.random.randint(0, 1000, size=5)
print(idx)
print(x_train_scaled[idx])
y_train[idx]

[326 328 840 732 233]
[[ 0.17769796  1.23336426 -0.10156759 -0.14386545 -0.48679983 -0.12497679
  -1.33627241  1.16594852]
 [-0.12601952  1.87416616 -0.10327644 -0.1416719  -0.52036942 -0.01608438
  -0.84637499  0.8263927 ]
 [ 0.65021841  0.4323619  -0.13819543 -0.03095488 -0.02952759 -0.04451932
  -0.8417093   0.64163439]
 [ 0.33046252 -1.0895426   0.28888462  0.01715829  0.52482427 -0.02024099
  -0.71106999  0.90129472]
 [ 0.88413416  1.3134645   0.43277185 -0.08091445 -0.43508452 -0.03793305
  -0.78105534  0.61666705]]


array([3.422, 1.664, 2.18 , 2.3  , 1.882])

In [61]:
# squeeze的作用
t=tf.constant([[1],[2],[3]])
print(t)
tf.squeeze(t,1)

tf.Tensor(
[[1]
 [2]
 [3]], shape=(3, 1), dtype=int32)


<tf.Tensor: shape=(3,), dtype=int32, numpy=array([1, 2, 3], dtype=int32)>

In [62]:
for i in range(10):
    print('helloworld',end='\r')

In [63]:
# 1. batch 遍历训练集 metric
#    1.1 自动求导
# 2. epoch结束 验证集 metric

epochs = 100 #多少次
batch_size = 32 
steps_per_epoch = len(x_train_scaled) // batch_size  #每一个epoch拿多少数据
print(steps_per_epoch)
optimizer = keras.optimizers.SGD()
metric = keras.metrics.MeanSquaredError()

362


In [64]:
#随机取数据,取出来32个样本
def random_batch(x, y, batch_size=32):
    idx = np.random.randint(0, len(x), size=batch_size)
    return x[idx], y[idx]

model = keras.models.Sequential([
    keras.layers.Dense(30, activation='relu',
                       input_shape=x_train.shape[1:]),
    keras.layers.Dense(1),
])
print(model.variables)

[<Variable path=sequential_2/dense_4/kernel, shape=(8, 30), dtype=float32, value=[[-3.53226781e-01 -8.71550143e-02  8.84665549e-02 -1.06190622e-01
  -1.81580216e-01  1.48594767e-01  2.68993407e-01 -1.59789830e-01
  -3.58466506e-02  1.85747653e-01  3.25623751e-02  1.31901532e-01
   2.35090107e-01  3.17431122e-01  3.61962169e-01  3.99379134e-02
  -3.38955373e-01  3.48113805e-01 -1.38914704e-01 -3.39161903e-01
   2.04154581e-01 -3.88600141e-01  1.78206056e-01 -1.27615422e-01
  -2.65605450e-02 -2.46898338e-01  3.16293865e-01  2.50627965e-01
  -3.74051034e-02 -2.60620475e-01]
 [ 2.87836105e-01 -1.32988572e-01  1.16138160e-02 -3.43860894e-01
  -6.36492074e-02 -2.82638013e-01 -2.11607695e-01  1.18857712e-01
   1.05623156e-01 -5.13508916e-02 -2.50442863e-01 -1.91720217e-01
  -1.31066442e-01  8.40168297e-02 -3.85582447e-01  3.64972353e-02
  -6.80407882e-02  2.98202425e-01 -6.93899393e-03 -2.27805972e-03
   3.16058546e-01 -2.89153457e-01 -9.84406769e-02  1.48670524e-01
  -3.42527658e-01  2.51477

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [65]:
#下面一部分相当于替代了fit函数
# print(model.variables)#可以看下这个里边有什么
# model.summary()
for epoch in range(epochs):#每一轮epochs训练所有的样本
    metric.reset_state()  #清空损失
    for step in range(steps_per_epoch):
        #随机取32个样本
        x_batch, y_batch = random_batch(x_train_scaled, y_train,
                                        batch_size)
        with tf.GradientTape() as tape:
            #得到预测值
            y_pred = model(x_batch) #等价于model.predict
 
            #删减了值为1的维度,二阶张量，变为一阶张量
            y_pred = tf.squeeze(y_pred, 1)
            #计算损失                                （标签值， 预测值）
            # loss = keras.losses.mean_squared_error(y_batch, y_pred)
            mse = tf.keras.losses.MeanSquaredError()
            loss = mse(y_batch, y_pred)
            #计算均方误差
            metric(y_batch, y_pred)
        #求梯度
        grads = tape.gradient(loss, model.variables)
        #梯度和变量绑定
        grads_and_vars = zip(grads, model.variables)
        #更新，通过grads，去更新模型的model.variables，也就是更新了w,b
        optimizer.apply_gradients(grads_and_vars)
        #打印，不要在循环内加print，影响\r
        p="Epoch "+str(epoch)+" train mse:"+str(metric.result().numpy())
        print(p, end='\r')
    print('') #打换行的目的是为了新起一行显示
    #搞了一波训练后，认为模型可以了，去验证集验证
    y_valid_pred = model(x_valid_scaled)
    # 删减了值为1的维度
#     print(y_valid_pred.shape)
    y_valid_pred = tf.squeeze(y_valid_pred, 1)
#     print(y_valid_pred.shape)
    # valid_loss = keras.losses.mean_squared_error(y_valid_pred, y_valid)  旧版本，弃用写法
    mse = tf.keras.losses.MeanSquaredError()
    valid_loss = mse(y_valid_pred, y_valid)
    print("\t", "valid mse: ", valid_loss.numpy())

Epoch 0 train mse:1.7855877
	 valid mse:  0.67304236
Epoch 1 train mse:0.49271628
	 valid mse:  0.4623937
Epoch 2 train mse:0.41052077
	 valid mse:  0.42695504
Epoch 3 train mse:0.40319397
	 valid mse:  0.4094533
Epoch 4 train mse:0.38953742
	 valid mse:  0.4033462
Epoch 5 train mse:0.39786083
	 valid mse:  0.38399863
Epoch 6 train mse:0.40499339
	 valid mse:  0.3930618
Epoch 7 train mse:0.36964643
	 valid mse:  0.3767504
Epoch 8 train mse:0.37788835
	 valid mse:  0.37278634
Epoch 9 train mse:0.34424084
	 valid mse:  0.36917853
Epoch 10 train mse:0.37092552
	 valid mse:  0.3699121
Epoch 11 train mse:0.34758037
	 valid mse:  0.3632957
Epoch 12 train mse:0.35260773
	 valid mse:  0.374298
Epoch 13 train mse:0.34365585
	 valid mse:  0.3636475
Epoch 14 train mse:0.35275793
	 valid mse:  0.36580238
Epoch 15 train mse:0.33667326
	 valid mse:  0.36123186
Epoch 16 train mse:0.34003085
	 valid mse:  0.36137885
Epoch 17 train mse:0.35100748
	 valid mse:  0.43883306
Epoch 18 train mse:0.34294382
	